# Importa Tudo

### Bibliotecas

In [1]:
import pandas as pd
import re

### Dados

In [26]:
import pandas as pd

# Specify the file path
iptu_filepath = r'D:\Drive\Codigos bee6\bee6_arquivos\bee6_geral\Raw\Download Iptus\IPTU_2024.csv'

# Load the IPTU data
try:
    iptu = pd.read_csv(iptu_filepath, encoding='latin1', on_bad_lines='skip', sep=';')
    print("IPTU data loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file at {iptu_filepath} was not found.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

C:\Users\guici\AppData\Local\Temp\ipykernel_23792\681783258.py:8: DtypeWarning: Columns (20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  iptu = pd.read_csv(iptu_filepath, encoding='latin1', on_bad_lines='skip', sep=';')


IPTU data loaded successfully.


In [29]:
# Filter the DataFrame to return rows where 'NOME DE LOGRADOURO DO IMOVEL' contains 'ROQUE PETRELLA'
roque_petrella_rows = iptu[iptu['NOME DE LOGRADOURO DO IMOVEL'].str.contains('ROQUE PETRELLA', case=False, na=False)]


In [33]:
# Filter the DataFrame to return rows where 'NOME DE LOGRADOURO DO IMOVEL' contains 'ROQUE PETRELLA'
# and 'NUMERO DO IMOVEL' is 46.0
roque_petrella_rows = iptu[
    iptu['NOME DE LOGRADOURO DO IMOVEL'].str.contains('ROQUE PETRELLA', case=False, na=False) & 
    (iptu['NUMERO DO IMOVEL'] == 46.0)
]


In [36]:
# Filter the DataFrame to return rows where:
# 1. 'NOME DE LOGRADOURO DO IMOVEL' contains 'ROQUE PETRELLA'
# 2. 'NUMERO DO IMOVEL' is 46.0
# 3. 'COMPLEMENTO DO IMOVEL' is 'CJ 604 E VG'
roque_petrella_rows = iptu[
    iptu['NOME DE LOGRADOURO DO IMOVEL'].str.contains('ROQUE PETRELLA', case=False, na=False) & 
    (iptu['NUMERO DO IMOVEL'] == 46.0) & 
    (iptu['COMPLEMENTO DO IMOVEL'] == 'CJ 604 E VG')
]


In [37]:
roque_petrella_rows

,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NUMERO DA NL,DATA DO CADASTRAMENTO,NUMERO DO CONDOMINIO,CODLOG DO IMOVEL,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,COMPLEMENTO DO IMOVEL,BAIRRO DO IMOVEL,...,ANO DA CONSTRUCAO CORRIGIDO,QUANTIDADE DE PAVIMENTOS,TESTADA PARA CALCULO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TIPO DE TERRENO,FATOR DE OBSOLESCENCIA,ANO DE INICIO DA VIDA DO CONTRIBUINTE,MES DE INICIO DA VIDA DO CONTRIBUINTE,FASE DO CONTRIBUINTE
1806693,0851000478-1,2024,1,01/01/24,04-3,13699-9,R ROQUE PETRELLA,46.0,CJ 604 E VG,BROOKLIN,...,2014.0,10,51.67,Escritório/consultório em condomínio (unidade ...,Comercial vertical - padrão C,De esquina,0.92,2015,1.0,0.0


### Trata Iptu

In [5]:
# Seleciona apenas o que é relevante
iptu = iptu[['TIPO DE CONTRIBUINTE 1','CPF/CNPJ DO CONTRIBUINTE 1','NOME DO CONTRIBUINTE 1',
             'TIPO DE CONTRIBUINTE 2','CPF/CNPJ DO CONTRIBUINTE 2','NOME DO CONTRIBUINTE 2',
             'NUMERO DO CONTRIBUINTE','BAIRRO DO IMOVEL','CEP DO IMOVEL','NOME DE LOGRADOURO DO IMOVEL',
             'NUMERO DO IMOVEL','COMPLEMENTO DO IMOVEL','REFERENCIA DO IMOVEL','AREA CONSTRUIDA','AREA DO TERRENO',
             'ANO DA CONSTRUCAO CORRIGIDO','VALOR DO M2 DE CONSTRUCAO','TIPO DE USO DO IMOVEL']]

# Cria o mapeamento para gerar tipo_imovel e tipo_uso

mapeamento = {
    "Apartamento em condomínio": ["Apartamento", "Residencial"],
    "Residência": ["Casa", "Residencial"],
    "Residência coletiva, exclusive cortiço (mais de uma residência no lote)": ["Casa", "Residencial"],
    "Garagem (unidade autônoma) em edifício em condomínio de uso exclusivamente residencial": ["Apartamento", "Residencial"],
    "Escritório/consultório em condomínio (unidade autônoma)": ["Sala Comercial", "Comercial"],
    "Terreno": ["Terreno", "Terreno"],
    "Residência e outro uso (predominância residencial)": ["Casa", "Residencial"],
    "Loja": ["Loja", "Comercial"],
    "Loja e residência (predominância comercial)": ["Loja", "Comercial"],
    "Garagem (unidade autônoma) em edifício em condomínio de escritórios, consultórios ou misto": ["Sala Comercial", "Comercial"],
    "Prédio de escritório ou consultório, não em condomínio, com ou sem lojas": ["Sala Comercial", "Comercial"],
    "Flat de uso comercial (semelhante a hotel)": ["Apartamento", "Comercial"],
    "Loja em edifício em condomínio (unidade autônoma)": ["Loja", "Comercial"],
    "Outras edificações de uso comercial, com utilização múltipla": ["Sala Comercial", "Comercial"],
    "Indústria": ["Galpao", "Comercial"],
    "Oficina": ["Galpao", "Comercial"],
    "Garagem (unidade autônoma) de prédio de garagens": ["Apartamento", "Comercial"],
    "Armazéns gerais e depósitos": ["Galpao", "Comercial"],
    "Escola": ["Casa", "Comercial"],
    "Templo": ["Casa", "Comercial"],
    "Flat residencial em condomínio": ["Apartamento", "Residencial"],
    "Outras edificações de uso de serviço, com utilização múltipla": ["Sala Comercial", "Comercial"],
    "Hotel, pensão ou hospedaria": ["Casa", "Comercial"],
    "Outras edificações de uso coletivo, com utilização múltipla": ["Sala Comercial", "Comercial"],
    "Garagem (exclusive em prédio em condomínio)": ["Apartamento", "Residencial"],
    "Prédio de apartamento, não em condomínio, de uso misto (apartamentos e escritórios e/ou consultórios), com ou sem loja (predominância residencial)": ["Apartamento", "Residencial"],
    "Posto de serviço": ["Terreno", "Comercial"],
    "Prédio de apartamento, não em condomínio, de uso exclusivamente residencial": ["Apartamento", "Residencial"],
    "Cortiço": ["Casa", "Residencial"],
    "Hospital, ambulatório, casa de saúde e assemelhados": ["Casa", "Comercial"],
    "Outras edificações de uso especial, com utilização múltipla": ["Sala Comercial", "Comercial"],
    "Asilo, orfanato, creche, seminário ou convento": ["Casa", "Residencial"],
    "Cinema, teatro, casa de diversão, clube ou congênere": ["Casa", "Comercial"],
    "Clube esportivo": ["Casa", "Comercial"],
    "Estação radioemissora, de televisão ou empresa jornalística": ["Sala Comercial", "Comercial"],
    "Prédio de escritório, não em condomínio, de uso misto (apartamentos e escritórios e/ou consultórios) com ou sem loja (predominância comercial)": ["Sala Comercial", "Comercial"]}

# Convertendo valores para o formato de título
iptu['NOME DO CONTRIBUINTE 1'] = iptu['NOME DO CONTRIBUINTE 1'].str.lower()
iptu['NOME DO CONTRIBUINTE 2'] = iptu['NOME DO CONTRIBUINTE 2'].str.lower()
iptu['BAIRRO DO IMOVEL'] = iptu['BAIRRO DO IMOVEL'].str.lower()

iptu['tipo_imovel'] = iptu['TIPO DE USO DO IMOVEL'].apply(lambda x: mapeamento.get(x, [None, None])[0])
iptu['tipo_uso'] = iptu['TIPO DE USO DO IMOVEL'].apply(lambda x: mapeamento.get(x, [None, None])[1])

iptu['municipio'] = 'Sao Paulo'
iptu['estado'] = 'SP'

iptu['logradouro'] = (iptu['NOME DE LOGRADOURO DO IMOVEL'].astype(str) + ' ' +
                      iptu['NUMERO DO IMOVEL'].astype(str) + ', ' +
                      iptu['COMPLEMENTO DO IMOVEL'].astype(str) + ' - ' +
                      iptu['REFERENCIA DO IMOVEL'].astype(str))

iptu['logradouro'] = iptu['logradouro'].str.title()

# Removendo possíveis vírgulas e convertendo para float
iptu['VALOR DO M2 DE CONSTRUCAO'] = iptu['VALOR DO M2 DE CONSTRUCAO'].str.replace(',','.').astype(float)

# Substituindo valores nas colunas TIPO DE CONTRIBUINTE 1 e TIPO DE CONTRIBUINTE 2
iptu['TIPO DE CONTRIBUINTE 1'] = iptu['TIPO DE CONTRIBUINTE 1'].replace({'PESSOA FISICA (CPF)': 'CPF', 'PESSOA JURIDICA (CNPJ)': 'CNPJ'})
iptu['TIPO DE CONTRIBUINTE 2'] = iptu['TIPO DE CONTRIBUINTE 2'].replace({'PESSOA FISICA (CPF)': 'CPF', 'PESSOA JURIDICA (CNPJ)': 'CNPJ'})

# Renomeando colunas
iptu = iptu.rename(columns={
    'TIPO DE CONTRIBUINTE 1': 'tipo_contribuinte_1',
    'CPF/CNPJ DO CONTRIBUINTE 1': 'cpf_cnpj_contribuinte_1',
    'NOME DO CONTRIBUINTE 1': 'nome_1',
    'TIPO DE CONTRIBUINTE 2': 'tipo_contribuinte_2',
    'CPF/CNPJ DO CONTRIBUINTE 2': 'cpf_cnpj_contribuinte_2',
    'NOME DO CONTRIBUINTE 2': 'nome_2',
    'NUMERO DO CONTRIBUINTE': 'numero_iptu',
    'BAIRRO DO IMOVEL': 'bairro',
    'CEP DO IMOVEL': 'cep',
    'AREA CONSTRUIDA': 'area_construida_m2',
    'AREA DO TERRENO': 'area_total_m2',
    'ANO DA CONSTRUCAO CORRIGIDO': 'data_construcao',
    'VALOR DO M2 DE CONSTRUCAO': 'valor_venal_m2_medio'})

# Selecionando colunas específicas
iptu = iptu[[
    'tipo_contribuinte_1','cpf_cnpj_contribuinte_1','tipo_contribuinte_2',
    'cpf_cnpj_contribuinte_2','nome_1','nome_2','numero_iptu','tipo_imovel','tipo_uso',
    'estado','municipio','bairro','cep','logradouro','area_construida_m2','area_total_m2','data_construcao',
    'valor_venal_m2_medio']]

KeyError: "['TIPO DE CONTRIBUINTE 1', 'CPF/CNPJ DO CONTRIBUINTE 1', 'NOME DO CONTRIBUINTE 1', 'TIPO DE CONTRIBUINTE 2', 'CPF/CNPJ DO CONTRIBUINTE 2', 'NOME DO CONTRIBUINTE 2'] not in index"

### Escolhe todos os proprietarios de SP com 5 imoveis ou mais (por bairro ou não)

In [4]:
# Primeiro, filtra o iptu apenas para os bairros relevantes (opcional)
#iptu = iptu.query("bairro == 'vila andrade' or bairro == 'brooklin' or bairro == 'campo belo' or bairro == 'moema'")

# Agrega por nome de proproetario para os nome 1
top_1 = iptu.query("tipo_contribuinte_1 == 'CPF'")[['nome_1','cpf_cnpj_contribuinte_1','bairro']]
top_1 = top_1.groupby(["nome_1", "cpf_cnpj_contribuinte_1"]).size().reset_index(name='quantidade').sort_values(by = 'quantidade', ascending = False).query("quantidade >= 5")
top_1.columns = ['nome','cpf','quantidade']

# Agrega por nome de proproetario para os nome 2
top_2 = iptu.query("tipo_contribuinte_2 == 'CPF'")[['nome_2','cpf_cnpj_contribuinte_2']]
top_2 = top_2.groupby(["nome_2", "cpf_cnpj_contribuinte_2"]).size().reset_index(name='quantidade').sort_values(by = 'quantidade', ascending = False).query("quantidade >= 5")
top_2.columns = ['nome','cpf','quantidade']

# Concatena os dataframes
top_props = pd.concat(objs = [top_1,top_2]).drop_duplicates(subset = ['nome','cpf'])

### Procura o nome de um proprietário específico

In [26]:
def separar_logradouro(logradouro):
    tipo_logradouro = logradouro.split(' ')[0]
    nome_logradouro_match = re.search(rf'{tipo_logradouro}\s(.*?)(?=\s\d)', logradouro)
    nome_logradouro = nome_logradouro_match.group(1).lower() if nome_logradouro_match else ''
    numero_imovel_match = re.search(r'\d+', logradouro)
    numero_imovel = numero_imovel_match.group() if numero_imovel_match else ''
    complemento = logradouro.split('.0, ')[1] if '.0, ' in logradouro else ''
    return pd.Series([tipo_logradouro, nome_logradouro, numero_imovel, complemento])

# Aplicando a função no dataframe
iptu[['tipo_logradouro', 'nome_logradouro', 'numero_imovel', 'complemento']] = iptu['logradouro'].apply(separar_logradouro)

# Lista de busca
busca_proprietarios = [
    "Jovina,241","Damasceno Vieira,544","Damasceno Vieira,122","Nelson Gama de Oliveira,905",
    "Jose Gonçalves,405","Jose Gonçalves,292","Nicola Rollo,151","Nicola Rollo,26",
    "Jose de Oliveira coelho,451","João Simões de Souza,391","Francisco Pessoa,575",
    "Carvalho de Freitas,323","Carvalho de Freitas,420"]

# Criar uma lista de condições
conditions = []

for item in busca_proprietarios:
    nome_logradouro, numero_imovel = item.split(',')
    condition = (iptu['nome_logradouro'].str.lower().str.contains(nome_logradouro.strip().lower(), case=False)) & (iptu['numero_imovel'].astype(str) == numero_imovel.strip())
    conditions.append(condition)

# Combinar todas as condições com OR
combined_condition = conditions[0]
for condition in conditions[1:]:
    combined_condition = combined_condition | condition

# Filtrar o DataFrame usando a condição combinada
filtered_df = iptu[combined_condition][['tipo_contribuinte_1','nome_1','tipo_contribuinte_2',
                                        'nome_2','municipio','bairro','cep','logradouro',
                                        'area_construida_m2','tipo_logradouro','nome_logradouro',
                                        'numero_imovel','complemento']]

### Escolhe proprietários específicos para subirem no directus a partir do IPTU

In [6]:
busca_nomes = ['rubens garcia filho','san ramon participacoes ltda']
regex_nomes = '|'.join(busca_nomes)

iptu_selecionado = iptu.query(
    "(`nome_1`.str.contains(@regex_nomes) or `nome_2`.str.contains(@regex_nomes))")

iptu_selecionado = iptu_selecionado[['numero_iptu','tipo_imovel','tipo_uso','municipio','cep',
                                     'logradouro','area_construida_m2','area_total_m2',
                                     'data_construcao','valor_venal_m2_medio','nome_1']]

iptu_selecionado = iptu_selecionado.rename(columns={
    'area_construida_m2': 'area_construida',
    'area_total_m2': 'area_total', 'nome_1': 'proprietario'})
# iptu_selecionado pronto para o directus

### Prepara imóveis específicos para subirem no directus a partir de planilha

In [4]:
# Prepara o valor venal médio agregado por cep
iptu_venal = iptu_atual[['CEP DO IMOVEL','VALOR DO M2 DE CONSTRUCAO']].groupby(by = 'CEP DO IMOVEL').mean().reset_index()
iptu_venal.columns = ['cep','valor_venal_m2_medio']

# Nesse caso, faço pros imoveis da bee brokers
imoveis_julio = imoveis_julio[['tipo_imovel','tipo_uso','municipio','cep',
                               'logradouro','area_construida','area_total',
                               'valor_iptu_anual','valor_condominio_mensal',
                               'num_quartos','num_vagas_est','num_banheiros',
                               'num_suites','proprietario']]

imoveis_julio['valor_iptu_anual'] = imoveis_julio['valor_iptu_anual'].str.replace('.','').str.replace(',','.').astype(float)
imoveis_julio['valor_condominio_mensal'] = imoveis_julio['valor_condominio_mensal'].str.replace('.','').str.replace(',','.').astype(float)
imoveis_julio = imoveis_julio.merge(iptu_venal, how = 'left', on = 'cep')
# imoveis_julio pronto para o directus